All CShaper training pairs plus Zenodo 6460303 frames 0–269 from three recordings. StarryNite one-based labels paired to zero-based TIFF indices. Zenodo supervision is positive only. Checkpoints follow cumulative two-epoch blocks. Fixed CShaper evaluation after epoch 8.
Revision 2 adds retries for temporary Zenodo errors, intermediate weight snapshots, and local Gaussian targets around Zenodo point labels, inspired by Linajea (https://doi.org/10.1038/s41587-022-01427-7).

All verified paired volumes from Samples04–17. Samples18–20 remain held out, using the historical 36-volume benchmark. Three passes through the full training cohort. The benchmark uses generated reference masks.

In [ ]:

import sys, subprocess, json, re, hashlib
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel','beautifulsoup4'],check=True)
import nibabel, requests
from bs4 import BeautifulSoup
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
def normalize(a):
    lo, hi = np.percentile(a, (1, 99.8))
    return np.clip((a.astype(np.float32)-lo)/max(1,hi-lo),0,1)
def block(a,b):
    return nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU(),
                         nn.Conv3d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU())
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc, self.deep, self.dec, self.head = block(1,8), block(8,16), block(24,8), nn.Conv3d(8,1,1)
    def forward(self,x):
        a=self.enc(x)
        b=self.deep(F.avg_pool3d(a,(1,2,2)))
        return self.head(self.dec(torch.cat([a,F.interpolate(b,size=a.shape[2:],mode='trilinear',align_corners=False)],1)))


assert torch.cuda.is_available(), 'Kaggle GPU required'
torch.manual_seed(42)
rng=np.random.default_rng(42)
out=Path('/kaggle/working')
cache=Path('/tmp/cshaper')
cache.mkdir(exist_ok=True)
train=[f'Sample{i:02d}' for i in range(4,18)]
test=['Sample18','Sample19','Sample20']

def listing(folder):
    response=requests.get('https://drive.google.com/embeddedfolderview',params={'id':folder},timeout=40)
    response.raise_for_status()
    entries={}
    for anchor in BeautifulSoup(response.text,'html.parser').select('a'):
        title=anchor.select_one('.flip-entry-title')
        match=re.search(r'/(?:folders|d)/([\w-]+)',anchor.get('href',''))
        if title and match:
            entries[title.get_text(strip=True)]=match[1]
    return entries

def frames(entries, sample, suffix):
    pattern=rf'{sample}_(\d+)_{suffix}\.nii\.gz'
    return {int(m[1]): key for name,key in entries.items() if (m:=re.fullmatch(pattern,name))}

def fetch(key):
    path=cache/f'{key}.nii.gz'
    if not path.exists():
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry
        with requests.Session() as session:
            session.mount('https://', HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2,
                status_forcelist=(429, 500, 502, 503, 504))))
            with session.get('https://drive.usercontent.google.com/download',
                             params=dict(id=key, export='download', confirm='t'),
                             stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                chunks = response.iter_content(1024 * 1024)
                head = next(chunks, b'')
                if not head.startswith(bytes((31, 139))):
                    raise ValueError(f'Drive did not serve a gzip file for {key}')
                staging = path.with_suffix('.part')
                with staging.open('wb') as output:
                    output.write(head)
                    for chunk in chunks:
                        output.write(chunk)
                staging.replace(path)
    return nibabel.load(path), hashlib.sha256(path.read_bytes()).hexdigest()

folders=listing('151pBI7d_JfkOz4eQI4R4F2u_cKtlftBi')
manifest={}
for sample in train+test:
    children=listing(folders[sample])
    images=frames(listing(children['RawNuc']),sample,'rawNuc')
    masks=frames(listing(children['SegNuc']),sample,'segNuc')
    paired=sorted(images.keys() & masks.keys())
    sample_count = len(paired) if sample in train else 12
    assert len(paired) >= sample_count, (sample,len(paired))
    chosen=[paired[i] for i in np.linspace(0,len(paired)-1,sample_count,dtype=int)]
    manifest[sample]=[dict(t=t,image=images[t],mask=masks[t]) for t in chosen]
    print(sample,len(paired),flush=True)
(out/'split.json').write_text(json.dumps(dict(train=train,test=test,frames=manifest),indent=2))

def load(row):
    a, row['image_sha256']=fetch(row['image'])
    b, row['mask_sha256']=fetch(row['mask'])
    assert a.shape==b.shape and len(a.shape)==3 and np.allclose(a.affine,b.affine,atol=1e-4), 'Image and mask grids differ'
    spacing=np.asarray(a.header.get_zooms()[::-1])
    image=normalize(np.asarray(a.dataobj).transpose(2,1,0))
    labels=np.asarray(b.dataobj).transpose(2,1,0)
    ids=np.unique(labels)
    points=np.asarray(ndi.center_of_mass(labels>0,labels,ids[ids>0])).reshape(-1,3)
    assert len(points), 'Empty nucleus mask'
    return image,points,spacing


import io,requests,zipfile
URL="https://zenodo.org/records/6460303/files/mskcc_confocal.zip?download=1"

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time

class Reader(io.RawIOBase):
    def __init__(self):
        self.session = requests.Session()
        self.session.mount('https://', HTTPAdapter(max_retries=Retry(
            total=6, backoff_factor=2, backoff_max=30,
            status_forcelist=(429,500,502,503,504))))
        with self.session.get(URL,headers={'Range':'bytes=0-15'},stream=True,timeout=(20,120)) as r:
            r.raise_for_status()
            if r.status_code != 206:
                raise ValueError('Zenodo did not honor the opening range')
            self.size = int(r.headers['Content-Range'].split('/')[-1])
        self.pos = 0

    def tell(self):
        return self.pos

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        self.pos = (0,self.pos,self.size)[whence] + offset
        return self.pos

    def read(self, n=-1):
        start = self.pos
        end = min(self.size,start+n if n>=0 else self.size)
        if end<=start:
            return b''
        for attempt in range(4):
            try:
                with self.session.get(URL+f'&range_start={start}',
                                      headers={'Range':f'bytes={start}-{end-1}'},
                                      stream=True,timeout=(20,120)) as r:
                    r.raise_for_status()
                    if r.status_code!=206 or not r.headers['Content-Range'].startswith(f'bytes {start}-{end-1}/'):
                        raise ValueError('Unexpected Zenodo byte range')
                    data=r.content
                if len(data)!=end-start:
                    raise IOError('Short Zenodo read')
                self.pos=end
                return data
            except (requests.RequestException, IOError) as exc:
                if attempt==3:
                    raise
                print(f'Retrying Zenodo range {start}: {type(exc).__name__}',flush=True)
                time.sleep(5*(attempt+1))

import csv
subprocess.run([sys.executable,'-m','pip','install','-q','tifffile','imagecodecs'],check=True)
import tifffile
zenodo = zipfile.ZipFile(Reader())
zen_cache = cache/'zenodo'
zen_cache.mkdir(exist_ok=True)
zen_rows=[]
for sample in range(1,4):
    name=f'mskcc_confocal_s{sample}'
    root=f'mskcc-confocal/{name}'
    for t in range(270):
        image=f'{root}/images/{name}_t{t:03d}.tif'
        label=f'{root}/tracks/nuclei/t{t+1:03d}-nuclei'
        zenodo.getinfo(image)
        zenodo.getinfo(label)
        zen_rows.append(dict(recording=name,t=t,image=image,label=label))
(out/'zenodo_manifest.json').write_text(json.dumps(zen_rows,indent=2))

def load_zen(row):
    path=zen_cache/f"{row['recording']}_{row['t']:03d}.npz"
    if not path.exists():
        raw=zenodo.read(row['image'])
        image=tifffile.imread(io.BytesIO(raw))
        assert image.shape==(41,512,512)
        records=csv.reader(io.StringIO(zenodo.read(row['label']).decode('utf8')))
        points=np.asarray([[float(r[7]),float(r[6]),float(r[5])] for r in records
                           if len(r)>=10 and int(r[1])>0 and r[9].strip()])
        assert points.ndim==2 and points.shape[1]==3 and np.isfinite(points).all()
        assert ((points>=0)&(points<np.array(image.shape))).all()
        np.savez_compressed(path,image=image,points=points)
    with np.load(path,allow_pickle=False) as data:
        return normalize(data['image']),data['points']

net=Detector().cuda()
opt=torch.optim.AdamW(net.parameters(),lr=0.001)
updates=0
source_updates={'CShaper':0,'Zenodo':0}
training_rows=[('CShaper',row) for sample in train for row in manifest[sample]]
training_rows.extend(('Zenodo',row) for row in zen_rows)

def advance(first,last):
    global updates
    for epoch in range(first,last+1):
        net.train()
        for index in rng.permutation(len(training_rows)):
            source,row=training_rows[index]
            if source=='Zenodo':
                image,points=load_zen(row)
            else:
                image,points,_=load(row)
            points=np.clip(points.round().astype(int),0,np.array(image.shape)-1)
            size=np.minimum(image.shape,(16,64,64))
            for _ in range(2):
                start=np.clip(points[rng.integers(len(points))]-size//2+rng.integers(-6,7,3),0,np.asarray(image.shape)-size)
                slices=tuple(slice(int(a),int(a+b)) for a,b in zip(start,size))
                local=points-start
                local=local[((local>=0)&(local<size)).all(axis=1)]
                target=np.zeros(tuple(size),dtype=np.float32)
                target[tuple(local.T)]=1
                target=ndi.maximum_filter(target,size=(1,3,3))
                x=torch.as_tensor(image[slices].copy(),dtype=torch.float32,device='cuda')[None,None]
                y=torch.as_tensor(target,device='cuda')[None,None]
                logits=net(x)
                if source=='Zenodo':
                    seeds = np.zeros(tuple(size), dtype=np.float32)
                    seeds[tuple(local.T)] = 1
                    heat = ndi.gaussian_filter(seeds, sigma=(0.7,1.4,1.4), truncate=2)
                    if not heat.any():
                        continue
                    heat /= heat.max()
                    target_heat = torch.as_tensor(heat,device='cuda')[None,None]
                    known = target_heat > 0.1
                    loss = 0.25*F.mse_loss(logits.sigmoid()[known], target_heat[known])
                else:
                    loss=F.binary_cross_entropy_with_logits(logits,y,pos_weight=x.new_tensor(30.))
                assert torch.isfinite(loss)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                updates+=1
                source_updates[source]+=1
                if updates % 2000 == 0:
                    staging = out/'latest.tmp'
                    torch.save(dict(model=net.state_dict(),optimizer=opt.state_dict(),
                                    epoch=epoch,updates=updates,source_updates=source_updates.copy()),staging)
                    staging.replace(out/'latest.pt')
                    print(dict(epoch=epoch,updates=updates,source_updates=source_updates),flush=True)
        print(dict(epoch=epoch,updates=updates,source_updates=source_updates),flush=True)

for stop in (2,4,6,8):
    advance(stop-1,stop)
    torch.save(dict(model=net.state_dict(),optimizer=opt.state_dict(),epoch=stop,updates=updates,
                    source_updates=source_updates.copy()),out/f'epoch_{stop}.pt')
    (out/'progress.json').write_text(json.dumps(dict(epoch=stop,updates=updates,source_updates=source_updates)))
net.eval()
results={}
with torch.inference_mode():
    for sample in test:
        rows=[]
        for row in manifest[sample]:
            image,points,spacing=load(row)
            scores=np.empty_like(image,dtype=np.float32)
            for start in range(0,len(image),8):
                lo,hi=max(0,start-4),min(len(image),start+12)
                end=min(len(image),start+8)
                x=torch.as_tensor(image[lo:hi],dtype=torch.float32,device='cuda')[None,None]
                scores[start:end]=net(x).sigmoid()[0,0,start-lo:end-lo].cpu().numpy()
            pred=peak_local_max(scores,threshold_abs=0.5,min_distance=2,exclude_border=False)
            d=cdist(points,pred)
            i,j=linear_sum_assignment(np.where(d<=3,d,(min(d.shape)+1)*4))
            rows.append(dict(t=row['t'],matched=int((d[i,j]<=3).sum()),annotated=len(points),predicted=len(pred)))
        results[sample]=rows
(out/'summary.json').write_text(json.dumps(dict(train=train,test=test,frames=manifest,updates=updates,match_gate_voxels=3,
    label_type='published generated nucleus masks',results=results),indent=2))
